In [8]:
import csv
import math
from pathlib import Path 


In [9]:
DATA_PATH = Path("data/agent_deposits.csv")


In [10]:
def load_column(path: Path, column: str) -> tuple[list[float], list[tuple[int, str]]]:
    valid_floats = []
    errors = []

    with open(path, mode='r', newline='', encoding='utf-8') as file:
        reader = csv.DictReader(file)

        for row_idx, row in enumerate(reader, start=2):
            raw_value = row.get(column)

            try:
                
                value = float(raw_value)
                valid_floats.append(value)

            except ValueError:
                errors.append((row_idx, raw_value))

    # return f"Valid values {valid_floats}", f"Invalid values {errors}"
    return valid_floats, errors



load_column(DATA_PATH, "transactions")

([100.0,
  89.0,
  125.0,
  96.0,
  63.0,
  68.0,
  132.0,
  73.0,
  111.0,
  73.0,
  97.0,
  109.0,
  62.0,
  60.0,
  87.0,
  86.0,
  66.0,
  120.0,
  108.0,
  110.0,
  113.0,
  69.0,
  132.0,
  140.0,
  85.0,
  94.0,
  103.0,
  71.0,
  99.0,
  102.0,
  61.0,
  112.0,
  75.0,
  77.0,
  91.0],
 [(7, ''), (10, ''), (14, ''), (17, 'abc'), (29, '')])

In [11]:

def mean(values: list[float]) -> float:
    if not values:
        raise ValueError("The list of values is empty")
    return sum(values)/len(values)


def median(values: list[float]) -> float:
    """Return the median of the values."""
    if not values:
        raise ValueError("The list of values is empty.")   
    sorted_values = sorted(values)
    n = len(sorted_values) 
    if n % 2 == 0:
        return (sorted_values[n // 2 - 1] + sorted_values[n // 2]) / 2
    else:
        return sorted_values[n // 2]  


def mode(values: list[float]) -> list[float]:
    """Return every value that occurs most often (a list, because there can be ties or none)."""
    frequency = {}
    for value in values:
        frequency[value] = frequency.get(value, 0) + 1
    max_frequency = max(frequency.values())
    modes = [value for value, freq in frequency.items() if freq == max_frequency]
    return modes 


print(f"Mean_Transactions: {mean(load_column(DATA_PATH, "transactions")[0]):.2f}")
print(f"Median_Transactions: {median(load_column(DATA_PATH, "transactions")[0])}")
print(f"Mode_Transactions: {mode(load_column(DATA_PATH, "transactions")[0])}")

Mean_Transactions: 93.11
Median_Transactions: 94.0
Mode_Transactions: [132.0, 73.0]


In [12]:

def variance(values: list[float], sample: bool = True) -> float:
    """Sample variance (n - 1) by default; population variance (n) when sample is False."""
    n = len(values)
    if sample and n < 2:
        raise ValueError("sample variance needs atleast 2 values")
  
    centre = mean(values)
    squared_deviations = [(value - centre) ** 2 for value in values]
    divisor = n - 1  if sample else n
    return sum(squared_deviations) / divisor


def std_dev(values: list[float], sample: bool = True) -> float:
    return math.sqrt(variance(values, sample=True))


print(f"Variance: {variance(load_column(DATA_PATH, "transactions")[0]):.2f}")
print(f"Std_deviation: {std_dev(load_column(DATA_PATH, "transactions")[0]):.2f}")

Variance: 502.57
Std_deviation: 22.42


In [13]:
def quartiles(values: list[float]) -> tuple[float, float, float]:
    """Return (Q1, Q2, Q3). Document which quartile method you chose."""
    if not values:
        raise ValueError("Cannot compute quartiles of an empty list.")
    sorted_values = sorted(values)
    n = len(sorted_values)
    q1 = n // 4
    q2 = n // 2
    q3 = 3 * n // 4
    return (sorted_values[q1], sorted_values[q2], sorted_values[q3])


def iqr_outliers(values: list[float], k: float = 1.5) -> dict:
    """Return the IQR, the lower and upper fences, and the values outside them."""
    if not values:
        raise ValueError("Cannot compute IQR outliers of an empty list.")
    q1, q2, q3 = quartiles(values)
    iqr = q3 - q1
    lower_fence = q1 - k * iqr
    upper_fence = q3 + k * iqr
    outliers = [x for x in values if x < lower_fence or x > upper_fence]
    return {
        "IQR": iqr,
        "Lower Fence": lower_fence,
        "Upper Fence": upper_fence,
        "Outliers": outliers
    }

print(f"Quartiles: {quartiles(load_column(DATA_PATH, "transactions")[0])}")
print(f"IQR Outliers: {iqr_outliers(load_column(DATA_PATH, "transactions")[0])}")

Quartiles: (73.0, 94.0, 110.0)
IQR Outliers: {'IQR': 37.0, 'Lower Fence': 17.5, 'Upper Fence': 165.5, 'Outliers': []}
